# Druggability Axis Analysis

### Goal
Identify existing therapeutic compounds that target the 181 and 23 core pan-cancer metastatic metabolic genes.

### Purpose
To cross-reference the highly conserved metastatic metabolic signature against DGIdb to find druggable vulnerabilities. This helps prioritize targets for potential drug repurposing.

### Interpretation
- **Druggable Targets:** Genes with known drug interactions. \n- **Clinical Relevance:** Overlap with FDA-approved drugs indicates immediate translational potential for treating metastasis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import os

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14

%load_ext autoreload
%autoreload 2

from IPython.display import display, HTML
from druggability_config import TARGET_GENES, OUTPUT_DIR, OUTPUT_BASENAME, ANALYSIS_SUFFIX
from query_dbs import compile_drug_databases
from query_depmap import analyze_depmap_synergy
from query_advanced_analysis import query_tractability, query_string_ppi

In [ ]:
# ==========================================
# ⚙️ USER PARAMETERS (Export Options)
# ==========================================
# Full Notebook HTML Report Export Toggle:
# - True  -> Automatically exports the entire notebook (Markdown, Code, Tables, Figures) to styled HTML
# - False -> Disables automatic HTML export
SAVE_AS_HTML = True  # DEFAULT: False. Change to True to export the entire notebook!

print(f"Analyzing Axis: {TARGET_GENES}")
print(f"Output Base: {OUTPUT_BASENAME}")

## 2. Cross-reference Genes against Drug Databases

**Purpose**: Identifies if the genes in the hypothesized axis already have FDA-approved drugs or compounds in clinical trials.  
**Interpretation**: If a gene is druggable, we can rapidly test repurposing existing drugs. If multiple genes in the axis are druggable, it opens the door to combination therapy.

In [ ]:
df_drugs = compile_drug_databases(TARGET_GENES)

if not df_drugs.empty:
    # Save to CSV
    csv_path = os.path.join(OUTPUT_DIR, f"{OUTPUT_BASENAME}_drug_targets.csv")
    df_drugs.to_csv(csv_path, index=False)
    print(f"Saved drug targets to {csv_path}")
    display(df_drugs.head(15))
else:
    print("No drug interactions found.")

## 3. Target Tractability Prediction (Open Targets)

**Purpose**: For targets lacking known drugs, we query computational models to predict if they are structurally "druggable" by small molecules or antibodies.  
**Interpretation**: High tractability scores indicate that a gene is a biologically viable target for *de novo* drug discovery, even if no drugs currently exist.

In [ ]:
df_trac = query_tractability(TARGET_GENES)
if not df_trac.empty:
    display(df_trac)
else:
    print("No tractability data found.")

## 4. Protein-Protein Interaction (STRING Database)

**Purpose**: Queries the STRING network to check if there is physical or functional evidence that these targets interact with each other.  
**Interpretation**: High combined scores (>0.4) strongly support the "axis" hypothesis, indicating these genes operate in the same complex or pathway rather than in isolation.

In [ ]:
df_ppi = query_string_ppi(TARGET_GENES)
if not df_ppi.empty:
    display(df_ppi)
else:
    print("No PPI network connections found between these genes.")

## 5. Synergistic Cell Death (DepMap Co-dependency)

**Purpose**: Uses Cancer Dependency Map (DepMap) CRISPR knockout data to calculate the correlation of gene effects across hundreds of cancer cell lines.  
**Interpretation**: 
- **Positive correlation (> 0.3)**: Suggests genes are in the same functional pathway; knocking out either has a similar effect.
- **Negative correlation**: Suggests potential synthetic lethality or compensatory mechanisms.

In [ ]:
corr_matrix = analyze_depmap_synergy(TARGET_GENES)

if corr_matrix is not None:
    display(corr_matrix)
else:
    print("Skipping correlation visualization.")

---

## 6. Export Full Notebook Report to HTML

Compiling this entire interactive notebook into a single publication-ready and highly interactive HTML report.

---
## Druggability of Pan-Cancer Conserved Genes
In addition to the specific GLS axis, we also query the DGIdb database for the strictly conserved **23 pan-cancer genes** (upregulated in all 5 cancers) and the broadly conserved **181 genes** (upregulated in $\ge$ 4 cancers).


In [ ]:
df_23 = pd.read_csv(os.path.join(OUTPUT_DIR, 'druggable_targets_23_genes.csv'))
print(f"Total drug interactions for 23 conserved genes: {len(df_23)}")
display(df_23.head(10))

In [ ]:
df_181 = pd.read_csv(os.path.join(OUTPUT_DIR, 'druggable_targets_181_genes.csv'))
print(f"Total drug interactions for 181 conserved genes: {len(df_181)}")
display(df_181.head(10))

In [ ]:
import subprocess
import sys
import os

notebook_filename = 'druggability_axis_analysis.ipynb'
output_base = 'druggability_axis_analysis_5MetCan_100k'
output_dir = os.path.join('..', 'output', 'druggability')
os.makedirs(output_dir, exist_ok=True)

jupyter_bin = os.path.join(os.path.dirname(sys.executable), 'jupyter')
if not os.path.exists(jupyter_bin): jupyter_bin = 'jupyter'

cmd_html = [jupyter_bin, "nbconvert", "--to", "html", notebook_filename, "--output-dir", output_dir, "--output", output_base]
res_html = subprocess.run(cmd_html, capture_output=True, text=True)

if res_html.returncode == 0:
    print(f"🎉 SUCCESS: Notebook successfully exported to '{os.path.join(output_dir, output_base)}.html'")
else:
    print("❌ HTML export failed.")
    print(res_html.stderr)
